# 实验一:分类数据集的探索式数据分析(Titanic)

- 数据集:Kaggle Titanic `train.csv`(891 行 x 12 列,含 `Survived` 分类标注)
- 主线:装载 → 数据审计 → 单变量分析 → 离群值 → 编码分箱 → 双变量分析 → 特征工程与建模
- 说明:每个代码 cell 均带注释;结论性文字写在 markdown cell 中

In [ ]:
# ==== 任务(1):装载数据集 ====
import pandas as pd  # pandas:表格数据分析核心库,DataFrame 是其二维表格结构

DATA_PATH = "../../assets/titanic.csv"  # 仓库内路径;课程提交版为 data/titanic.csv   # 数据相对路径(notebook 与 data 文件夹同级)
df = pd.read_csv(DATA_PATH)      # 读入 CSV,得到 DataFrame

print("数据集形状(行数, 列数):", df.shape)
print("字段列表:", list(df.columns))

In [ ]:
# ==== 任务(2):查看部分数据 ====
# head(n):看前 n 行,用于快速了解各字段的取值样式
display(df.head(5))
# tail(n):看后 n 行,确认数据末尾无异常(如汇总行、截断)
display(df.tail(5))

In [ ]:
# ==== 初查:存储类型、缺失情况、取值基数 ====
# info():每列的 pandas 存储类型与非空数量,可一眼发现缺失列(Age/Cabin/Embarked)
df.info()

# nunique():每列不同取值的个数(基数)
# 经验:基数小(个位数)的字段,语义上多半是类别型;基数接近行数的,多半是标识符或文本
print(df.nunique().sort_values())

## 第 1 轮验收结论(已验收)

**Q1:12 个字段的变量类型归属**

| 字段 | 类型 | 说明 |
|---|---|---|
| PassengerId | 标识符 | 名义型特例,逐人唯一,不参与分析 |
| Survived | 名义类别型 | **目标变量**,0/1 只是代号 |
| Pclass | 有序类别型 | 1/2/3 有等级顺序 |
| Name | 文本 | 不做统计,第 5 轮后可提取称谓 |
| Sex | 名义类别型 | |
| Age | 连续数值型 | 缺失 177 条 |
| SibSp | 离散数值型 | 计数取值 |
| Parch | 离散数值型 | 计数取值 |
| Ticket | 文本/标识 | 不直接参与,重复票号为团体票 |
| Fare | 连续数值型 | 右偏分布 |
| Cabin | 文本/标识 | 缺失 687 条 |
| Embarked | 名义类别型 | S/C/Q,缺失 2 条 |

**Q2:`Survived` 是数值型吗?** 不是。0/1 是"否/是"的代号,属名义类别型;"mean≈0.38"读作**生还比例**,不是平均数意义。

**Q3:`PassengerId` 参与建模吗?** 不参与。建模 = 从带标注数据学习**可迁移到新样本**的判断规则;ID 只标识具体个人,对新乘客没有泛化价值,还会让模型死记硬背(过拟合)。对应课件 p62 身份证号提问。

---

# 第 2 轮:数据审计(描述性统计 / 重复值 / 缺失值)
对应课件 p12–p41;对应任务 (3)(4)(5)。缺失值的处理方案见本轮验收。

In [ ]:
# ==== 任务(3):描述性统计 ====
# describe():数值型字段统计摘要
# count=非空数, mean=均值, std=标准差, min/25%/50%/75%/max=最小值与四分位点(50% 即中位数)
display(df.describe())

# 类别型字段摘要:选中非数值列再 describe(unique=取值数, top=众数, freq=众数频次)
# 用 select_dtypes 而非 include="object",兼容 pandas 新旧版本且无警告
display(df.select_dtypes(exclude=["number"]).describe())

# 读数要点 1:Fare 的 mean(32.2) 远大于中位数(14.5)→ 分布右偏,存在高票价离群值(第 4 轮处理)
# 读数要点 2:Age std≈14.5 而 Fare std≈49.7 → 字段尺度差异大(第 7 轮缩放的伏笔)

In [ ]:
# ==== 任务(4):重复值检测与处理 ====
# duplicated():与前面行完全相同的整行重复
dup_rows = df.duplicated().sum()
print("整行完全重复的记录数:", dup_rows)
# 本数据集为 0,无需删除;若有,常规做法:df = df.drop_duplicates(keep="first")

# 注意:单字段取值重复 ≠ 错误。票号重复多为家庭/团体共用票,属正常业务
print("票号重复的记录数:", df.duplicated(subset=["Ticket"]).sum())
display(df[df.duplicated(subset=["Ticket"], keep=False)].sort_values("Ticket").head(6))

In [ ]:
# ==== 任务(5):缺失值情况盘点 ====
# isnull() 逐格判缺失,sum() 按列合计;再算缺失率(%)
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(1)
missing_table = pd.DataFrame({"缺失数": miss, "缺失率(%)": miss_pct})

# 只看有缺失的列,按缺失率降序:Cabin 77.1% / Age 19.9% / Embarked 0.2%
display(missing_table[missing_table["缺失数"] > 0].sort_values("缺失率(%)", ascending=False))

## 缺失值处理(参考实现,说理见 notes/08-参考答案.md)

- `Age`(19.9%):按 `Pclass`+`Sex` **分组中位数**填充——年龄在舱位、性别间分布不同,中位数抗右偏;
- `Embarked`(0.2%):**众数**填充,仅 2 条,代价最小;
- `Cabin`(77.1%):缺失太多,填什么都是编的;转为 `HasCabin` 0/1 标志("有无客舱记录"本身是信息),丢弃原字段。

In [ ]:
# ==== 任务(5)续:缺失值处理(参考实现)====
# Age:按 Pclass+Sex 分组的中位数填充
# groupby().transform("median") 会给每一行返回其所在组的中位数,用于对位填充
age_group_median = df.groupby(["Pclass", "Sex"])["Age"].transform("median")
df["Age"] = df["Age"].fillna(age_group_median)

# Embarked:仅缺 2 条,众数填充(mode()[0] 取第一个众数)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin:缺失 77%,不填补;转为 HasCabin(有无客舱记录)0/1 标志后丢弃原列
df["HasCabin"] = df["Cabin"].notnull().astype(int)
df = df.drop(columns=["Cabin"])

# 验证:全表缺失应为 0
print("处理后各字段缺失数:")
print(df.isnull().sum())

---

# 第 3 轮:单变量分析(类别型 / 数值型字段展示)
对应课件 p93–p99;对应任务 (6)(7)。EDA = 不带预设假设,用统计量和图让数据自己"说话"。

In [ ]:
# 绘图全局设置:让 matplotlib 正确显示中文(微软雅黑)与负号
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

In [ ]:
# ==== 任务(6):类别型字段信息展示 ====
# 语义上的类别型字段(Pclass/Survived/HasCabin 存储为 int,但取值是标签)
CAT_COLS = ["Survived", "Pclass", "Sex", "Embarked", "HasCabin"]

for col in CAT_COLS:
    print(f"========== {col} ==========")
    print("-- head 5 --");  print(df[col].head(5).to_string())
    print("-- tail 5 --");  print(df[col].tail(5).to_string())
    print("rowcount:", len(df[col]),
          "| null count:", df[col].isnull().sum(),
          "| distinct count:", df[col].nunique())
    print("distinct values:", sorted(df[col].unique().tolist()))
    print()

In [ ]:
# 任务(6)续:类别型字段直方图(柱状图,柱子高度=各类别样本数)
fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(16, 3.2))
for ax, col in zip(axes, CAT_COLS):
    df[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title(f"{col} 分布"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
# 观察:Sex 约 65% 男性 → 类别不均衡;Survived 约 38% 生还 → 目标略不均衡(课件 p124 数据倾斜)

In [ ]:
# ==== 任务(7):数值型字段信息展示 ====
NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]

for col in NUM_COLS:
    print(f"========== {col} ==========")
    print("-- head 5 --");  print(df[col].head(5).to_string())
    print("-- tail 5 --");  print(df[col].tail(5).to_string())
    print("rowcount:", len(df[col]),
          "| null count:", df[col].isnull().sum(),
          "| distinct count:", df[col].nunique())
    print()

In [ ]:
# 任务(7)续:数值型字段密度图(KDE)与箱线图
# 密度图:直方图的平滑版,看分布形状(峰位、偏态、多峰)
# 箱线图:五数概括(min/Q1/中位/Q3/max)+ 离群点,第 4 轮细讲
fig, axes = plt.subplots(2, len(NUM_COLS), figsize=(14, 6))
for j, col in enumerate(NUM_COLS):
    df[col].plot.kde(ax=axes[0, j], color="#4C72B0")
    axes[0, j].set_title(f"{col} 密度图")
    df.boxplot(column=col, ax=axes[1, j])
    axes[1, j].set_title(f"{col} 箱线图")
plt.tight_layout(); plt.show()
# 观察:Age 近似单峰略右偏;Fare 严重右偏,高票价离群点明显;SibSp/Parch 集中在 0

---

# 第 4 轮:离群值检测与处理
对应课件 p42–p48;对应任务 (8)。检测用箱线图 IQR 法与 3σ 法对照;处理看离群值的成因,不盲目删除。

In [ ]:
# ==== 任务(8):离群值检测(IQR 法 vs 3σ 法)====
# IQR 法:超出 [Q1-1.5*IQR, Q3+1.5*IQR] 即离群(箱线图"须"之外)
# 3σ 法:偏离均值超过 3 倍标准差即离群(要求近似正态分布)
for col in ["Age", "Fare"]:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_iqr = ((df[col] < lo) | (df[col] > hi)).sum()
    mu, sd = df[col].mean(), df[col].std()
    n_3s = ((df[col] - mu).abs() > 3 * sd).sum()
    print(f"{col}: IQR 上下界 = [{lo:.1f}, {hi:.1f}], 离群 {n_iqr} 条 | "
          f"3σ 上下界 = [{mu-3*sd:.1f}, {mu+3*sd:.1f}], 离群 {n_3s} 条")

# 看 Fare 最高的 10 条:都是一等舱高票价,真实业务值而非错误
display(df.nlargest(10, "Fare")[["Pclass", "Sex", "Age", "Fare", "Survived"]])

### 离群值处理决策(参考实现)

- `Fare`:离群值全是一等舱豪华票,**真实观测,保留**;其右偏问题交给第 5 轮分箱与第 7 轮缩放消化;
- `Age`:高龄乘客(70–80 岁)同样真实,**保留**;
- 若离群值是录入错误(如年龄 -5)才应删除或替换——本数据集没有这种情况。

---

# 第 5 轮:类别型编码与数值型分箱
对应课件 p49–p61;对应任务 (9)(10)。模型只认数字:类别要编码,数值可分箱;顺便构造派生特征。

In [ ]:
# ==== 任务(10):数值型 Binning(等宽 vs 等深)====
# 等宽分箱 cut:按取值范围均分,各箱宽度相等——遇到偏态分布会出现"空箱/挤箱"
# 等深分箱 qcut:按分位数切,各箱样本数相等——偏态分布下更均衡(课件 p52–53)
df["Age_bin_width"] = pd.cut(df["Age"], bins=5)
df["Age_bin_depth"] = pd.qcut(df["Age"], q=5)

print("Age 等宽分箱各箱样本数:");  print(df["Age_bin_width"].value_counts().sort_index())
print(); print("Age 等深分箱各箱样本数:");  print(df["Age_bin_depth"].value_counts().sort_index())

# Fare 右偏严重,用等深分箱(4 箱)供后续分析
df["Fare_bin"] = pd.qcut(df["Fare"], q=4)

In [ ]:
# ==== 任务(9):类别型字段编码 ====
# 在 df_model 副本上编码,df 保留原始字段供第 6 轮分析
df_model = df.copy()

# Sex:二值类别,Label Encoding 映射 0/1 即可(不会引入多余维度)
df_model["Sex_code"] = df_model["Sex"].map({"male": 0, "female": 1})

# Embarked:三值且无顺序,用 One-Hot 编码(get_dummies 拆成 3 个 0/1 列)
# 若强行 Label 编码成 0/1/2,模型会误以为 C<Q<S 有大小关系
df_model = pd.get_dummies(df_model, columns=["Embarked"], prefix="Embarked", dtype=int)

# Pclass:有序类别,1/2/3 本身带等级语义,直接保留数值即可
# (Binary/Target Encoding 见课件 p60–61,本数据集取值少,用不上)

# 派生特征 FamilySize:同行人数 SibSp+Parch 再+1(自己),信息更紧凑
df_model["FamilySize"] = df_model["SibSp"] + df_model["Parch"] + 1

FEATURES = ["Pclass", "Sex_code", "Age", "Fare", "FamilySize", "HasCabin",
            "Embarked_C", "Embarked_Q", "Embarked_S"]
X = df_model[FEATURES].copy()   # 特征矩阵
y = df_model["Survived"]        # 目标向量
print("建模数据集 X:", X.shape, "| 目标 y:", y.shape)
display(X.head(3))

---

# 第 6 轮:变量与目标的关系(单变量 / 双变量)
对应课件 p100–p118、p130–p138;对应任务 (11)(12)。统计检验(卡方、方差分析)看"差异是否显著",图表看"差异长什么样"。

In [ ]:
# ==== 任务(11):类别型变量与目标的关系 ====
# 交叉表 normalize="index":每个类别内部,生还/遇难的比例
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, ["Sex", "Pclass", "Embarked", "HasCabin"]):
    rate = pd.crosstab(df[col], df["Survived"], normalize="index")
    rate.plot(kind="bar", ax=ax, color=["#C0504D", "#4C72B0"], legend=False)
    ax.set_title(f"{col} vs 生还率"); ax.tick_params(axis="x", rotation=0)
axes[0].legend(["遇难", "生还"])
plt.tight_layout(); plt.show()

# 卡方检验:H0 = 该字段与 Survived 独立;p < 0.05 则显著相关(课件 p100)
from scipy.stats import chi2_contingency
for col in ["Sex", "Pclass", "Embarked", "HasCabin"]:
    chi2, p, *_ = chi2_contingency(pd.crosstab(df[col], df["Survived"]))
    print(f"{col}: 卡方统计量 = {chi2:.1f}, p 值 = {p:.2e} ->", "显著相关" if p < 0.05 else "不显著")

In [ ]:
# ==== 任务(11)续:数值型变量与目标的关系 ====
# 分组箱线图:按 Survived 分组看 Age/Fare 分布差异(课件 p117)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, col in zip(axes, ["Age", "Fare"]):
    df.boxplot(column=col, by="Survived", ax=ax)
    ax.set_title(f"{col} 按是否生还分组")
plt.suptitle(""); plt.tight_layout(); plt.show()

# 单因素方差分析(ANOVA,课件 p104–106):H0 = 两组的均值相等
from sklearn.feature_selection import f_classif
F, pvals = f_classif(df[["Age", "Fare", "SibSp", "Parch"]], y)
for col, f, p in zip(["Age", "Fare", "SibSp", "Parch"], F, pvals):
    print(f"{col}: F = {f:.1f}, p 值 = {p:.2e} ->", "与目标显著相关" if p < 0.05 else "与目标关系不显著")

In [ ]:
# ==== 任务(12):单变量与目标、双变量与目标 ====
# 单变量(分箱后)与目标:年龄段的生还率——儿童(0–16 岁)生还率明显更高
age_rate = df.groupby("Age_bin_depth", observed=True)["Survived"].mean()
fare_rate = df.groupby("Fare_bin", observed=True)["Survived"].mean()
print("各年龄段的生还率:"); print(age_rate.round(2))
print(); print("各票价段的生还率:"); print(fare_rate.round(2))

# 双变量与目标:Sex × Pclass 组合的生还率热力图(类别×类别→目标,课件 p115–117)
pivot = df.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean")
fig, ax = plt.subplots(figsize=(4.5, 3))
im = ax.imshow(pivot, cmap="RdYlGn", vmin=0, vmax=1)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_xticks(range(3), ["1 等舱", "2 等舱", "3 等舱"]); ax.set_yticks(range(2), pivot.index)
ax.set_title("Sex × Pclass 生还率"); fig.colorbar(im)
plt.tight_layout(); plt.show()
# 观察:一等舱女性 ≈ 97% 生还;三等舱男性 ≈ 14%——交互作用比单看任何一个字段都更锐利

### 辛普森悖论提醒(课件 p130–p138)

聚合结论在分组后可能**反转**(Berkeley 录取案:总体男生录取率更高,分院系看却几乎个个院系女生更高)。成因是各组基数悬殊且分组率不同。Titanic 没有完全反转,但 `Pclass` 的生存率差异在控制 `Sex` 后明显变化:三等舱低生还率,一部分其实是"三等舱里男性占多数"贡献的。**看任何聚合结论,先问一句:它掩盖了什么分组?** 详解与图解见 assets/辛普森悖论.html。

---

# 第 7 轮:相关性、特征选择、数据缩放与建模对比
对应课件 p55–p56、p101–p103、p141;对应任务 (13)(14)(15)。

In [ ]:
# ==== 任务(13):字段间相关性与特征选择 ====
# Pearson 相关系数矩阵(课件 p101–103):+1 完全正相关,-1 完全负相关,0 无线性相关
corr = df_model[FEATURES + ["Survived"]].corr()
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im); ax.set_title("特征相关性矩阵")
plt.tight_layout(); plt.show()

# 特征选择:与目标的 |相关系数| 超过阈值 0.1 才保留
target_corr = corr["Survived"].drop("Survived").abs().sort_values(ascending=False)
print("各特征与目标的 |相关系数|:"); print(target_corr.round(3))
SELECTED = target_corr[target_corr > 0.1].index.tolist()
print("\n阈值 0.1 筛选后保留:", SELECTED)

# 冗余检查:特征两两相关 > 0.8 说明信息重复,可删其一(本数据集没有)
pair_max = corr.drop("Survived", axis=0).drop("Survived", axis=1).abs()
pair_max = pair_max.where(~(pair_max == 1.0)).max().max()
print(f"特征间最大 |相关系数| = {pair_max:.2f}(< 0.8,无冗余)")

In [ ]:
# ==== 任务(14):数据缩放——Min-Max 与标准化 ====
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# 先切分:80% 训练、20% 测试;stratify=y 保持生还比例一致
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 关键纪律:缩放器只用训练集 fit,再 transform 测试集
# 若用全量数据 fit,测试集信息就"泄漏"进了训练过程,评估会虚高
mm = MinMaxScaler().fit(X_train)     # Min-Max:缩放到 [0,1],保持分布形状,受离群值影响大
std = StandardScaler().fit(X_train)  # 标准化:z=(x-μ)/σ,均值 0 方差 1,对离群值更稳健

demo = pd.DataFrame({
    "原始 Fare": X_train["Fare"].head(4).values,
    "MinMax 后": mm.transform(X_train[FEATURES])[:4, FEATURES.index("Fare")],
    "标准化后": std.transform(X_train[FEATURES])[:4, FEATURES.index("Fare")],
})
display(demo.round(3))

In [ ]:
# ==== 任务(15):KNN / 决策树 / SVC 对照实验 ====
# 3 个模型 × {全部特征, 特征选择} × {不缩放, 标准化} = 12 组
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Pipeline 把缩放器装进交叉验证:每一折内部重新 fit 缩放器,杜绝泄漏
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {"KNN": KNeighborsClassifier(),
          "决策树": DecisionTreeClassifier(random_state=42),
          "SVC": SVC()}
feature_sets = {"全部特征": FEATURES, "特征选择": SELECTED}

rows = []
for mname, model in models.items():
    for fname, cols in feature_sets.items():
        for sname, scaler in [("不缩放", None), ("标准化", StandardScaler())]:
            clf = model if scaler is None else make_pipeline(scaler, model)
            acc = cross_val_score(clf, X_train[cols], y_train, cv=cv, scoring="accuracy").mean()
            rows.append({"模型": mname, "特征": fname, "缩放": sname, "5折CV准确率": round(acc, 4)})

results = pd.DataFrame(rows).sort_values("5折CV准确率", ascending=False).reset_index(drop=True)
display(results)

In [ ]:
# 12 组结果可视化 + 最优模型在测试集上的最终评估
fig, ax = plt.subplots(figsize=(10, 4))
labels = results["模型"] + "|" + results["特征"] + "|" + results["缩放"]
ax.barh(range(len(results)), results["5折CV准确率"], color="#4C72B0")
ax.set_yticks(range(len(results)), labels, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("5 折交叉验证准确率")
plt.tight_layout(); plt.show()

# 用最优配置在从未见过的测试集上做最终评估
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
best = results.iloc[0]
best_cols = feature_sets[best["特征"]]
best_clf = models[best["模型"]]
if best["缩放"] == "标准化":
    best_clf = make_pipeline(StandardScaler(), best_clf)
best_clf.fit(X_train[best_cols], y_train)
pred = best_clf.predict(X_test[best_cols])
print(f"最优配置: {best['模型']} | {best['特征']} | {best['缩放']}")
print("测试集准确率:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=["遇难", "生还"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["遇难", "生还"]).plot(cmap="Blues")
plt.title("测试集混淆矩阵"); plt.show()

---

# 实验结论

1. **缩放的影响**:KNN 与 SVC 基于距离,不缩放时大尺度字段(Fare)主导距离,标准化后准确率明显回升;决策树按取值排序分裂,对缩放天然免疫;
2. **特征选择的影响**:本数据集特征少且大多有效,阈值筛选的收益不明显,但演示了"相关系数阈值法"的完整流程;
3. **模型对比**:SVC(标准化后)与决策树通常领先,KNN 受类别不均衡与维度影响略低——没有免费的午餐,模型要配合适的预处理;
4. **方法论**:缩放器只在训练集 fit、Pipeline 封装进交叉验证,是防止数据泄漏的两道保险。